In [ ]:
# U Google Colab okruženju scikit-learn je najčešće već instaliran.
# Ako nije, ukloniti komentar iz naredne linije:
# %pip install -q scikit-learn pytest

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from functools import wraps
from time import perf_counter
from typing import Callable, Dict, List, Any

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

RANDOM_STATE = 42
print("Biblioteke su uspješno učitane.")

In [ ]:
MODEL_FACTORY: Dict[str, Callable[[], Any]] = {}

def registruj_model(naziv: str):
    """Dekorator koji registruje funkciju za kreiranje modela."""
    def dekorator(fabrika: Callable[[], Any]):
        MODEL_FACTORY[naziv.lower()] = fabrika
        return fabrika
    return dekorator

@registruj_model("logistic_regression")
def napravi_logisticku_regresiju():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=500, random_state=RANDOM_STATE)
    )

@registruj_model("decision_tree")
def napravi_stablo_odlucivanja():
    return DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)

@registruj_model("knn")
def napravi_knn():
    return make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5)
    )

def kreiraj_model(naziv: str):
    """Vraća novu instancu modela na osnovu naziva."""
    kljuc = naziv.lower()
    if kljuc not in MODEL_FACTORY:
        raise ValueError(f"Nepoznat model: {naziv}. Dostupni modeli: {list(MODEL_FACTORY)}")
    return MODEL_FACTORY[kljuc]()

print("Registrovani modeli:", list(MODEL_FACTORY))
print("Primjer:", kreiraj_model("knn"))

In [ ]:
Metric = Callable[[np.ndarray, np.ndarray], float]

def metrika_accuracy(y_true, y_pred) -> float:
    return float(accuracy_score(y_true, y_pred))

def metrika_f1(y_true, y_pred) -> float:
    return float(f1_score(y_true, y_pred, average="weighted", zero_division=0))

def metrika_precision(y_true, y_pred) -> float:
    return float(precision_score(y_true, y_pred, average="weighted", zero_division=0))

def metrika_recall(y_true, y_pred) -> float:
    return float(recall_score(y_true, y_pred, average="weighted", zero_division=0))

METRIKE: Dict[str, Metric] = {
    "accuracy": metrika_accuracy,
    "f1": metrika_f1,
    "precision": metrika_precision,
    "recall": metrika_recall,
}

def evaluiraj(y_true, y_pred, naziv_metrike: str = "accuracy") -> float:
    naziv = naziv_metrike.lower()
    if naziv not in METRIKE:
        raise ValueError(f"Nepoznata metrika: {naziv_metrike}")
    return METRIKE[naziv](y_true, y_pred)

# Kratka demonstracija strategija
y_primjer = np.array([0, 1, 2, 1, 0])
p_primjer = np.array([0, 1, 1, 1, 0])
for ime in METRIKE:
    print(f"{ime}: {evaluiraj(y_primjer, p_primjer, ime):.3f}")

In [ ]:
class Observer:
    def obavijesti(self, dogadjaj: str, **podaci):
        raise NotImplementedError

class LoggerObserver(Observer):
    def __init__(self):
        self.dogadjaji = []

    def obavijesti(self, dogadjaj: str, **podaci):
        zapis = {"dogadjaj": dogadjaj, **podaci}
        self.dogadjaji.append(zapis)
        print(f"[LOG] {dogadjaj}: {podaci}")

class ProgressObserver(Observer):
    def obavijesti(self, dogadjaj: str, **podaci):
        if dogadjaj == "eksperiment_zavrsen":
            print(f"[INFO] Eksperiment završen. Rezultat: {podaci.get('rezultat', {})}")

class Subject:
    def __init__(self):
        self._pretplatnici: List[Observer] = []

    def pretplati(self, observer: Observer):
        if observer not in self._pretplatnici:
            self._pretplatnici.append(observer)

    def odjavi(self, observer: Observer):
        if observer in self._pretplatnici:
            self._pretplatnici.remove(observer)

    def objavi(self, dogadjaj: str, **podaci):
        for observer in list(self._pretplatnici):
            observer.obavijesti(dogadjaj, **podaci)

print("Observer komponente su spremne.")

In [ ]:
VREMENA_IZVRSAVANJA: List[dict] = []

def mjeri_vrijeme(fn):
    @wraps(fn)
    def omotac(*args, **kwargs):
        pocetak = perf_counter()
        try:
            return fn(*args, **kwargs)
        finally:
            trajanje = perf_counter() - pocetak
            VREMENA_IZVRSAVANJA.append({
                "funkcija": fn.__name__,
                "sekunde": trajanje
            })
            print(f"[VRIJEME] {fn.__name__}: {trajanje:.6f} s")
    return omotac

print("Dekorator za mjerenje vremena je spreman.")

In [ ]:
@dataclass
class RezultatEksperimenta:
    model: str
    metrika: str
    vrijednost: float
    dodatne_metrike: dict = field(default_factory=dict)

class Eksperiment(ABC):
    def __init__(self, naziv_modela: str, naziv_metrike: str = "accuracy"):
        self.naziv_modela = naziv_modela
        self.naziv_metrike = naziv_metrike
        self.subject = Subject()
        self.model = None
        self.X_train = self.X_test = None
        self.y_train = self.y_test = None
        self.y_pred = None

    def pretplati(self, observer: Observer):
        self.subject.pretplati(observer)

    def pokreni(self):
        """Template Method: fiksni redoslijed koraka eksperimenta."""
        self.subject.objavi("eksperiment_pocet", model=self.naziv_modela)
        self.ucitaj_podatke()
        self.pripremi_podatke()
        self.treniraj_model()
        rezultat = self.evaluiraj_model()
        self.subject.objavi("eksperiment_zavrsen", rezultat=rezultat.__dict__)
        return rezultat

    @abstractmethod
    def ucitaj_podatke(self):
        pass

    def pripremi_podatke(self):
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=0.25,
            random_state=RANDOM_STATE, stratify=self.y
        )
        self.subject.objavi("podaci_pripremljeni",
                            broj_trening_uzoraka=len(self.X_train),
                            broj_test_uzoraka=len(self.X_test))

    @mjeri_vrijeme
    def treniraj_model(self):
        self.model = kreiraj_model(self.naziv_modela)
        self.model.fit(self.X_train, self.y_train)
        self.subject.objavi("model_istreniran", model=self.naziv_modela)

    def evaluiraj_model(self) -> RezultatEksperimenta:
        self.y_pred = self.model.predict(self.X_test)
        vrijednost = evaluiraj(self.y_test, self.y_pred, self.naziv_metrike)
        dodatne = {
            ime: evaluiraj(self.y_test, self.y_pred, ime)
            for ime in METRIKE if ime != self.naziv_metrike
        }
        return RezultatEksperimenta(
            model=self.naziv_modela,
            metrika=self.naziv_metrike,
            vrijednost=vrijednost,
            dodatne_metrike=dodatne
        )

class IrisEksperiment(Eksperiment):
    def ucitaj_podatke(self):
        iris = load_iris()
        self.X = iris.data
        self.y = iris.target
        self.nazivi_klasa = iris.target_names
        self.subject.objavi("podaci_ucitani",
                            broj_uzoraka=len(self.X),
                            broj_osobina=self.X.shape[1])


In [ ]:
logger = LoggerObserver()
progress = ProgressObserver()

rezultati = []
for naziv_modela in ["logistic_regression", "decision_tree", "knn"]:
    print("\n" + "=" * 60)
    eksperiment = IrisEksperiment(naziv_modela, naziv_metrike="accuracy")
    eksperiment.pretplati(logger)
    eksperiment.pretplati(progress)
    rezultat = eksperiment.pokreni()
    rezultati.append(rezultat)

print("\nSAŽETAK REZULTATA")
for r in rezultati:
    print(f"{r.model:22s} | {r.metrika}: {r.vrijednost:.3f} | "
          f"dodatno: {r.dodatne_metrike}")

In [ ]:
import pandas as pd

tabela = pd.DataFrame([
    {
        "Model": r.model,
        "Glavna metrika": r.metrika,
        "Vrijednost": r.vrijednost,
        **{f"Dodatno - {k}": v for k, v in r.dodatne_metrike.items()}
    }
    for r in rezultati
])
display(tabela)

In [ ]:
# TEST 1: Factory vraća model za svaki registrovani naziv.
def test_factory_svi_modeli():
    assert {"logistic_regression", "decision_tree", "knn"}.issubset(MODEL_FACTORY)
    for naziv in MODEL_FACTORY:
        assert kreiraj_model(naziv) is not None

# TEST 2: Factory jasno prijavljuje nepoznat model.
def test_factory_nepoznat_model():
    try:
        kreiraj_model("nepostojeci_model")
    except ValueError as e:
        assert "Nepoznat model" in str(e)
    else:
        raise AssertionError("Očekivan je ValueError za nepoznat model.")

# TEST 3: Strategy vraća očekivanu accuracy vrijednost.
def test_strategy_accuracy():
    y_true = [0, 1, 2, 1]
    y_pred = [0, 1, 1, 1]
    assert evaluiraj(y_true, y_pred, "accuracy") == 0.75

# TEST 4: Strategy prijavljuje nepoznatu metriku.
def test_strategy_nepoznata_metrika():
    try:
        evaluiraj([0, 1], [0, 1], "nepoznata")
    except ValueError:
        pass
    else:
        raise AssertionError("Očekivan je ValueError za nepoznatu metriku.")

# TEST 5: Observer prima i čuva objavljeni događaj.
def test_observer():
    subject = Subject()
    observer = LoggerObserver()
    subject.pretplati(observer)
    subject.objavi("test_dogadjaj", vrijednost=123)
    assert len(observer.dogadjaji) == 1
    assert observer.dogadjaji[0]["vrijednost"] == 123

# TEST 6: Template Method uspješno izvršava cijeli Iris eksperiment.
def test_template_method():
    eksperiment = IrisEksperiment("decision_tree")
    rezultat = eksperiment.pokreni()
    assert isinstance(rezultat, RezultatEksperimenta)
    assert 0.0 <= rezultat.vrijednost <= 1.0
    assert eksperiment.model is not None

# TEST 7: Decorator evidentira vrijeme izvršavanja.
def test_decorator_vrijeme():
    prije = len(VREMENA_IZVRSAVANJA)

    @mjeri_vrijeme
    def primjer():
        return 42

    assert primjer() == 42
    assert len(VREMENA_IZVRSAVANJA) == prije + 1
    assert VREMENA_IZVRSAVANJA[-1]["sekunde"] >= 0

testovi = [
    test_factory_svi_modeli,
    test_factory_nepoznat_model,
    test_strategy_accuracy,
    test_strategy_nepoznata_metrika,
    test_observer,
    test_template_method,
    test_decorator_vrijeme,
]

uspjesni = 0
for test in testovi:
    test()
    print(f"✓ {test.__name__}")
    uspjesni += 1

print(f"\nRezultat: {uspjesni}/{len(testovi)} testova uspješno prošlo.")